# Lab 1 — OpenCV Intro

This notebook covers the following operations:

1. Open image  
2. Open video  
3. Histograms  
4. Edge detection  
5. Filtering  
6. Morphological operations  

**Important note for Lightning Studio:** graphical windows opened with `cv2.imshow()` may not work reliably in a browser notebook.  
In this notebook we use **Matplotlib** to display images and video frames instead.

---

## Before you start

Make sure the `material/` folder is available (images and optional videos).  
If you are using a published Studio template, you should already have it.

## 📦 Requirements and environment setup

In this laboratory we rely on a small set of core Python libraries that are widely used in computer vision and scientific computing.
If any of these libraries are missing, you can install them by running the following cell.

In [ ]:
!pip install opencv-python numpy matplotlib

First, we import all the necessary libraries used throughout the notebook.

Then, we define the main project directories:

- The data folder, which contains all input materials (images, videos, etc.)

- The output folder, where all generated results will be saved

Make sure the `material/` and `saves/` folders are available.  

If you are using a published Studio template, you should already have them.

In [ ]:
# Imports
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Matplotlib defaults
plt.rcParams["figure.figsize"] = (10, 5)

MATERIAL_DIR = "material"
SAVE_DIR = "saves"


In [ ]:
# Adjust these paths if needed
MATERIAL_DIR = "../material"  # matches the original scripts structure

PATH_HOME = os.path.join(MATERIAL_DIR, "home.jpg")
PATH_EDGE = os.path.join(MATERIAL_DIR, "edge.jpg")
PATH_LOGO = os.path.join(MATERIAL_DIR, "opencv_logo.jpg")
PATH_J = os.path.join(MATERIAL_DIR, "j.png")
PATH_OPENING = os.path.join(MATERIAL_DIR, "opening.png")
PATH_CLOSING = os.path.join(MATERIAL_DIR, "closing.png")

def show_bgr(img_bgr, title=None):
    """Display a BGR image (OpenCV default) using Matplotlib (expects RGB)."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.imshow(img_rgb)
    if title:
        plt.title(title)
    plt.axis("off")

def show_gray(img_gray, title=None):
    plt.imshow(img_gray, cmap="gray")
    if title:
        plt.title(title)
    plt.axis("off")


## 1) Open an image (read, display, optionally save)

In this section, we load an image using **OpenCV**.

In [ ]:
# Open and display an image

PATH_HOME = os.path.join(MATERIAL_DIR, "home.jpg")
img = cv2.imread(PATH_HOME)
assert img is not None, f"Could not read image at: {PATH_HOME}"

print("Image shape (H, W, C):", img.shape)
print("Image dtype:", img.dtype)

- `os.path.join(...)` builds the full path to the image.
- ``cv2.imread()`` reads the image from disk.
When OpenCV loads an image, it stores it as a NumPy array with shape: ``(H, W, C)``

Where:

- H = height

- W = width

- C = number of channels (3 for a color image)

Important notes:

- OpenCV loads color images in BGR format, not RGB.

- The data type is typically ``uint8``, meaning pixel values range from 0 to 255.

- The assert statement ensures that the image was loaded correctly.

OpenCV provides ``cv2.imshow()``, which opens the image in a separate graphical window outside the notebook interface. This may not work in browser-based environments such as Lightning Studio. For this reason, we use **Matplotlib**. Matplotlib expects RGB format. If we display an OpenCV image directly with `plt.imshow()`, the colors will look incorrect (for example, blue and red will be swapped).  
For this reason, we must convert the image from **BGR to RGB** before visualization.


In [ ]:
def show_bgr(img_bgr, title=None):
    """Display a BGR image (OpenCV default) using Matplotlib (expects RGB)."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.imshow(img_rgb)
    if title:
        plt.title(title)
    plt.axis("off")

plt.figure()
show_bgr(img, "Home (BGR → RGB for display)")
plt.show()

- ``cv2.cvtColor()`` converts the image from BGR to RGB.
- ``plt.imshow()`` displays the correctly formatted image.
- ``plt.axis("off")`` removes axis ticks for a cleaner visualization.
- ``plt.figure()`` creates a new figure before displaying the image.

In this section, we save the loaded image to the output directory. ``cv2.imwrite()`` writes the image (NumPy array).

In [ ]:
out_path = os.path.join(SAVE_DIR, "image_copy.jpg")
cv2.imwrite(out_path, img)
print("Saved to:", out_path)

## 2) Open a video (capture, read frames, display)

In OpenCV, video input is handled through the ``cv2.VideoCapture`` class. This class provides a unified interface to acquire video frames either from a camera device or from a video file. Once a VideoCapture object is created, frames can be read sequentially and processed using the same operations applied to images.
When ``cv2.VideoCapture`` is initialized with an integer value (for example ``cv2.VideoCapture(0)``), OpenCV attempts to open a camera device. The integer identifies the camera index, where 0 usually refers to the default webcam. This mode enables real-time video acquisition and is commonly used when OpenCV is executed locally on a machine with direct access to hardware devices.
Alternatively, ``cv2.VideoCapture`` can be initialized with a file path (for example ``cv2.VideoCapture``("Video.mp4")). In this case, frames are read from a stored video file. From a processing point of view, there is no difference between camera input and file input: each frame is returned as an image and can be analyzed or transformed in the same way.
In cloud-based environments such as Lightning Studio, direct access to local hardware devices (including webcams) is not supported. The code runs on a remote machine that has no connection to the camera attached to your laptop. For this reason, initializing ``cv2.VideoCapture(0)`` in Lightning Studio will typically fail or cause the kernel to crash.
For this laboratory, video input is therefore handled using video files.

In [ ]:
# Try to open a local video file first (recommended in Lightning)
VIDEO_PATH = os.path.join(MATERIAL_DIR, "Video.mp4")  # change if needed

if os.path.exists(VIDEO_PATH):
    cap = cv2.VideoCapture(VIDEO_PATH)
    source = VIDEO_PATH
else:
    # Fallback: webcam (usually works only on a local machine, not in cloud)
    cap = cv2.VideoCapture(0)
    source = "webcam (0)"

if not cap.isOpened():
    raise RuntimeError(f"Cannot open video source: {source}")

print("Opened video source:", source)

# Read and display a few frames
frames_to_show = 5
shown = 0

plt.figure(figsize=(12, 8))
while shown < frames_to_show:
    ret, frame = cap.read()
    if not ret:
        print("End of stream or cannot receive frame.")
        break
    shown += 1
    plt.subplot(2, 3, shown)
    show_bgr(frame, f"Frame {shown}")
plt.tight_layout()
plt.show()

cap.release()

We start by opening the video:
1. ``cv2.VideoCapture()`` initializes a video stream.
2. If a local video file exists, we open it (recommended for cloud environments like Lightning).
3. Otherwise, we attempt to open the webcam using index 0.
4. ``cap.isOpened()`` verifies that the video source was successfully initialized.

Then, we display video Frames:
1. `` cap.read()`` returns:
    - ``ret``: ``True`` if a frame was successfully read
2. frame: the image frame (as a NumPy array)
3. Each frame is treated like a standard OpenCV image (H × W × C).
4. We display the first few frames using Matplotlib.
5. plt.subplot() arranges multiple frames in a grid.
6. cap.release() frees the video resource after use.

## 3) Histograms

We will compute and visualize three types of histograms:

- **Grayscale histogram** — distribution of pixel intensities (0..255).
- **RGB channel histograms** — histograms for each channel (B, G, R) to show per-channel contributions.
- **Histogram on a region (ROI)** — computed using a binary mask to analyze only a selected area.

Histograms are useful for:
  - spotting under/over-exposure,
  - choosing thresholds for segmentation,
  - detecting color biases or dominant tones.

**What to observe**
- The grayscale histogram shows how brightness values are distributed (shadows vs highlights).
- RGB histograms show how each color channel contributes to overall appearance.
- The masked histogram reflects only the pixels inside the ROI and can differ significantly from the full-image histogram.


In [ ]:
# Grayscale histogram (home.jpg)
img_gray = cv2.imread(PATH_HOME, cv2.IMREAD_GRAYSCALE)
assert img_gray is not None, f"Could not read image at: {PATH_HOME}"

hist = cv2.calcHist([img_gray], [0], None, [256], [0, 256])

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
show_gray(img_gray, "Home (grayscale)")
plt.subplot(1,2,2)
plt.plot(hist)
plt.title("Grayscale histogram")
plt.xlabel("Pixel intensity")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

- `cv2.imread(PATH_HOME, cv2.IMREAD_GRAYSCALE)` reads the image as a 2D NumPy array (`H × W`) with `dtype` usually `uint8`.
- `cv2.calcHist([img_gray], [0], None, [256], [0, 256])` computes the frequency of each intensity bin (0–255).
- Plotting the histogram shows frequency (y) vs intensity (x).

In [ ]:
# RGB histograms
img_color = cv2.imread(PATH_HOME)
assert img_color is not None, f"Could not read image at: {PATH_HOME}"

b, g, r = cv2.split(img_color)

hist_b = cv2.calcHist([b], [0], None, [256], [0, 256])
hist_g = cv2.calcHist([g], [0], None, [256], [0, 256])
hist_r = cv2.calcHist([r], [0], None, [256], [0, 256])

plt.figure(figsize=(8, 5))
plt.plot(hist_r, label="Red channel")
plt.plot(hist_g, label="Green channel")
plt.plot(hist_b, label="Blue channel")
plt.title("RGB histograms")
plt.xlabel("Pixel intensity")
plt.ylabel("Frequency")
plt.legend()
plt.show()

- `cv2.imread(PATH_HOME)` reads a color image as `H × W × C` in **BGR** order (OpenCV default).
- `b, g, r = cv2.split(img_color)` separates the channels.
- `cv2.calcHist([channel], [0], None, [256], [0, 256])` computes the histogram for each channel.
- Plotting the three histograms together reveals channel dominance and color distribution.

In [ ]:
# Histogram from a region (mask)
img_gray = cv2.imread(PATH_HOME, cv2.IMREAD_GRAYSCALE)
assert img_gray is not None, f"Could not read image at: {PATH_HOME}"

mask = np.zeros(img_gray.shape[:2], np.uint8)
mask[100:300, 100:400] = 255

masked_img = cv2.bitwise_and(img_gray, img_gray, mask=mask)

hist_full = cv2.calcHist([img_gray], [0], None, [256], [0, 256])
hist_mask = cv2.calcHist([img_gray], [0], mask, [256], [0, 256])

plt.figure(figsize=(12,6))
plt.subplot(2,2,1); show_gray(img_gray, "Original (grayscale)")
plt.subplot(2,2,2); show_gray(mask, "Mask")
plt.subplot(2,2,3); show_gray(masked_img, "Masked image")
plt.subplot(2,2,4)
plt.plot(hist_full, label="Full image")
plt.plot(hist_mask, label="Masked region")
plt.title("Histogram: full vs masked region")
plt.xlim([0,256])
plt.legend()
plt.tight_layout()
plt.show()

- Create a binary mask: `mask = np.zeros(img_gray.shape[:2], np.uint8)` and set ROI to 255 (`mask[100:300, 100:400] = 255`).
- `masked_img = cv2.bitwise_and(img_gray, img_gray, mask=mask)` applies the mask to the image.
- `cv2.calcHist([img_gray], [0], mask, [256], [0, 256])` computes the histogram only for pixels where the mask is non-zero.
- Compare `hist_full` and `hist_mask` to see how the ROI differs from the whole image.

## 4) Edge Detection — Sobel Operator

In this section, we detect edges using the **Sobel operator**, a gradient-based method that highlights intensity changes in an image.

Edges correspond to regions where pixel intensity changes rapidly — in other words, where the image gradient is high.


In [ ]:
# Edge detection with Sobel (based on edge.jpg)
img = cv2.imread(PATH_EDGE, cv2.IMREAD_GRAYSCALE)
assert img is not None, f"Could not read image at: {PATH_EDGE}"

- The image is loaded as a 2D NumPy array (H × W).
- Edge detection is typically applied to grayscale images because we are interested in intensity variation, not color.

In [ ]:
img_blur = cv2.GaussianBlur(img, (3,3), 0)

Before computing gradients, we smooth the image:
- Reduces noise
- Prevents small fluctuations from being detected as edges
- Improves stability of gradient computation

This is a standard preprocessing step in edge detection.

In [ ]:
sobelx = cv2.Sobel(img_blur, cv2.CV_64F, 1, 0, ksize=5)
sobely = cv2.Sobel(img_blur, cv2.CV_64F, 0, 1, ksize=5)
sobelxy = cv2.Sobel(img_blur, cv2.CV_64F, 1, 1, ksize=5)

plt.figure(figsize=(12,8))
plt.subplot(2,2,1); show_gray(img_blur, "Original (blurred)")
plt.subplot(2,2,2); show_gray(sobelxy, "Sobel XY")
plt.subplot(2,2,3); show_gray(sobelx, "Sobel X")
plt.subplot(2,2,4); show_gray(sobely, "Sobel Y")
plt.tight_layout()
plt.show()

The Sobel operator approximates the **first-order partial derivatives** of the image intensity function.
1. ``sobelx`` → horizontal derivative (detects vertical edges)
2. ``sobely`` → vertical derivative (detects horizontal edges)
3. ``sobelxy`` → combined derivative

If we model a grayscale image as a function $I(x, y)$, then the Sobel operator computes approximations of:

$$
\frac{\partial I}{\partial x}
\quad \text{and} \quad
\frac{\partial I}{\partial y}
$$

Edges correspond to locations where the intensity changes rapidly. The gradient vector is:

$$
\nabla I =
\begin{bmatrix}
\frac{\partial I}{\partial x} \\
\frac{\partial I}{\partial y}
\end{bmatrix}
$$

Its magnitude is:

$$
|\nabla I| =
\sqrt{
\left(\frac{\partial I}{\partial x}\right)^2 +
\left(\frac{\partial I}{\partial y}\right)^2
}
$$

- If the magnitude is small → the region is flat.
- If the magnitude is large → there is a strong edge.

**Why Edges Appear Where Derivatives Are Large?** An edge is a region where pixel intensity changes abruptly.

For example:
- Flat region → pixel values are similar → derivative ≈ 0  
- Sharp boundary → pixel values change quickly → derivative is large  

The Sobel operator detects these strong intensity transitions by approximating the derivatives using convolution kernels, which dimension is ``ksize``. 
Larger ``ksize`` values mean: Stronger smoothing effect, More robust gradients, Less sensitivity to noise

Smaller ``ksize`` values mean: Sharper edge localization, Higher sensitivity to noise
There is no single “correct” ``ksize`` value, it depends on the image and your goal. 

As general guidelines:
- If the image is noisy → increase ksize
- If you need precise edge localization → use smaller ksize
- In most computer vision pipelines → ksize=3 is sufficient

## 5) Filtering (2D convolution, averaging, blur)

We will apply:
- a custom averaging kernel using `filter2D`,
- a simple box blur using `blur` (and optionally Gaussian blur).

What you should observe:
- Filtering reduces noise but also reduces detail.
- Strong smoothing removes high-frequency content (fine textures) and softens edges.

### Filter2D with a custom kernel
$$
    K = \frac{1}{25}
    \begin{bmatrix}
    1 & 1 & 1 & 1 & 1 \\
    1 & 1 & 1 & 1 & 1 \\
    1 & 1 & 1 & 1 & 1 \\
    1 & 1 & 1 & 1 & 1 \\
    1 & 1 & 1 & 1 & 1
    \end{bmatrix}
$$
- kernel = np.ones((5,5), np.float32) / 25 creates a normalized averaging kernel. Normalization (/25) ensures the output brightness stays comparable to the original.
- cv2.filter2D(img, -1, kernel) convolves the image with the kernel. The -1 means the output has the same depth as the source.

The operation works like this: keep this kernel above a pixel, add all the 25 pixels below this kernel, take the average, and replace the central pixel with the new average value. This operation is continued for all the pixels in the image.

In [ ]:
# Filtering with a custom kernel (opencv_logo.jpg)
img = cv2.imread(PATH_LOGO)
assert img is not None, f"Could not read image at: {PATH_LOGO}"

kernel = np.ones((5,5), np.float32) / 25
dst = cv2.filter2D(img, -1, kernel)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1); show_bgr(img, "Original")
plt.subplot(1,2,2); show_bgr(dst, "Averaging (filter2D)")
plt.tight_layout()
plt.show()

### Blurring

Image blurring is achieved by convolving the image with a low-pass filter kernel. It is useful for removing noise. It actually removes high frequency content (eg: noise, edges) from the image. 

**Averaging** is done by convolving an image with a normalized box filter. 

$$
K = \frac{1}{9}
\begin{bmatrix}
1 & 1 & 1 \\
1 & 1 & 1 \\
1 & 1 & 1
\end{bmatrix}
$$


It simply takes the average of all the pixels under the kernel area and replaces the central element. This is done by the function ``cv.blur()``

In [ ]:
# Blur (box filter)
img = cv2.imread(PATH_LOGO)
assert img is not None, f"Could not read image at: {PATH_LOGO}"

blur = cv2.blur(img, (3,3))
# gaussian = cv2.GaussianBlur(img, (5,5), 0)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1); show_bgr(img, "Original")
plt.subplot(1,2,2); show_bgr(blur, "Blurred (box blur 3x3)")
plt.tight_layout()
plt.show()



**Gaussian Blur** uses a Gaussian kernel, instead of a box filter, to smooth the image. This operation is performed using the function ``cv.GaussianBlur()``. The kernel size (width and height) must be positive odd numbers (e.g., 3×3, 5×5, 7×7). The function also requires the standard deviation of the Gaussian distribution along the X and Y directions, specified by sigmaX and sigmaY. Gaussian blurring performs a weighted average of neighboring pixels, giving more importance to pixels closer to the center of the kernel. Because of this property, it is particularly effective at reducing Gaussian noise while preserving image structure better than a simple box filter.

In [ ]:
# Blur (box filter)
img = cv2.imread(PATH_LOGO)
assert img is not None, f"Could not read image at: {PATH_LOGO}"

gaussian = cv2.GaussianBlur(img, (5,5), 0)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1); show_bgr(img, "Original")
plt.subplot(1,2,2); show_bgr(blur, "Blurred (box blur 3x3)")
plt.tight_layout()
plt.show()

## 6) Morphological operations (binary images)

Morphological operations are used to process **binary images**, where pixels typically represent:

- Foreground (object) → white (255)
- Background → black (0)

These operations are based on a **structuring element (kernel)** that defines how pixels are examined in a local neighborhood.

In this section, we visualize:

- **Erosion**
- **Dilation**
- **Opening** (erosion → dilation)
- **Closing** (dilation → erosion)



In [ ]:
kernel = np.ones((5,5), np.uint8)

The kernel (also called structuring element):
- Defines the neighborhood size (5×5 here)
- Controls how aggressively the morphology modifies shapes
- Larger kernels produce stronger effects

### Erosion & Dilation

In [ ]:
# Erosion / dilation on j.png
img = cv2.imread(PATH_J, cv2.IMREAD_GRAYSCALE)
assert img is not None, f"Could not read image at: {PATH_J}"

erosion = cv2.erode(img, kernel, iterations=1)
dilation = cv2.dilate(img, kernel, iterations=1)

plt.figure(figsize=(12,5))
plt.subplot(1,3,1); show_gray(img, "Original")
plt.subplot(1,3,2); show_gray(erosion, "Erosion")
plt.subplot(1,3,3); show_gray(dilation, "Dilation")
plt.tight_layout()
plt.show()

**Erosion** reduces the size of white (foreground) regions in a binary image. Conceptually, a pixel remains white only if all the pixels covered by the structuring element (kernel) are also white. If even one pixel in that neighborhood is black (background), the central pixel is set to black. As a result, foreground objects gradually shrink. Thin structures become thinner, small protrusions are removed, and tiny isolated white regions may disappear completely. This makes erosion particularly useful for removing small foreground noise and refining object boundaries. However, because erosion reduces object size, applying it too aggressively (large kernel or multiple iterations) can remove important details or even eliminate small objects entirely.

**Dilation** expands white (foreground) regions in a binary image. A pixel becomes white if at least one pixel under the structuring element (kernel) is white. In other words, if any pixel in the neighborhood belongs to the foreground, the central pixel is turned white.
As a consequence, foreground objects grow in size. Small gaps between nearby components may close, thin structures become thicker, and small holes inside objects may shrink. Dilation is therefore useful for strengthening object regions and reconnecting fragmented parts. However, excessive dilation can cause objects to merge unintentionally or lose fine structural details due to over-expansion.

### Opening

In [ ]:
# Opening
img = cv2.imread(PATH_OPENING, cv2.IMREAD_GRAYSCALE)
assert img is not None, f"Could not read image at: {PATH_OPENING}"

opening = cv2.morphologyEx(img, cv2.MORPH_OPEN, kernel)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1); show_gray(img, "Original (opening)")
plt.subplot(1,2,2); show_gray(opening, "After opening")
plt.tight_layout()
plt.show()

**Opening** is a morphological operation obtained by applying *erosion followed by dilation* using the same structuring element. First, erosion shrinks the foreground regions, removing small white noise and thin protrusions. Then, dilation expands the remaining regions back to approximately their original size. However, the small components that were removed during erosion do not reappear. As a result, opening is particularly useful for removing small foreground noise while preserving the overall shape and size of larger objects. It smooths object contours and breaks thin connections between components without significantly altering major structures.

### Closing

In [ ]:
# Closing
img = cv2.imread(PATH_CLOSING, cv2.IMREAD_GRAYSCALE)
assert img is not None, f"Could not read image at: {PATH_CLOSING}"

closing = cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1); show_gray(img, "Original (closing)")
plt.subplot(1,2,2); show_gray(closing, "After closing")
plt.tight_layout()
plt.show()

**Closing** is the opposite sequence: *dilation followed by erosion*, again using the same structuring element. First, dilation expands the foreground regions, filling small holes and connecting nearby components. Then, erosion reduces the regions back toward their original size. However, small holes that were filled during dilation typically remain filled. As a result, closing is useful for filling small gaps, closing narrow breaks, and removing small holes inside foreground objects while preserving the overall object size and structure.

**Morphological operations** are not based on intensity gradients (like Sobel), but on shape and structure. They are extremely useful after thresholding, edge detection and segmentation. They refine binary masks and improve object quality.